# Stream distance: compute at 0.5 m, then resample and mask at each resolution

This workflow does:
1. Build 0.5 m Euclidean distance rasters from buffered stream/ditch vectors.
2. Resample the 0.5 m distances to 20 m (average) and mask by the 20 m stream raster.
3. Independently resample the 0.5 m distances to 40/80/160 m (average) and mask by each matching-resolution stream raster.

In [1]:
import os
import numpy as np
import rasterio
import tools
from dotenv import load_dotenv

load_dotenv()
krycklan_data_path = os.getenv('KRYCKLAN_DATA_PATH')
krycklan_gis_data_path = os.getenv('KRYCKLAN_GIS_DATA_PATH')
print(f'Krycklan data path: {krycklan_data_path}')
print(f'Krycklan GIS data path: {krycklan_gis_data_path}')

file_ends = ['channels', '5haStreams']
coarse_resos = [40, 80, 160]

catchs = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13,
                   141, 142, 143,
                   151, 152, 153,
                   161, 162, 163, 164,
                   20, 21, 22])

nodata_val = -9999.0

Krycklan data path: /Users/jpnousu/Library/CloudStorage/OneDrive-Luonnonvarakeskus/Krycklan data
Krycklan GIS data path: /Users/jpnousu/Data/Krycklan_GIS_data


In [2]:
# Step 0: Build 0.5 m Euclidean distance rasters from buffered vectors
import sys
import geopandas as gpd

sys.path.append('/Applications/WhiteboxTools_darwin_m_series/WBT')
from whitebox_tools import WhiteboxTools
wbt = WhiteboxTools()

buffer_distance = 0.5

streams_dir = os.path.join(krycklan_data_path, 'GIS', 'CATCHMENTS', 'Krycklan_5ha_streams')
ditches_dir = os.path.join(krycklan_data_path, 'GIS', 'CATCHMENTS', 'ManualStreams', 'PROCESSED')
cleaned_dem_fp = os.path.join(krycklan_data_path, 'GIS', 'DEM', 'Krycklan_2015_DEM_0.5m', 'Krycklan_2015_DEM_cleaned.tif')

streams_fp = os.path.join(streams_dir, '5haStreams.shp')
out_fp_streams = os.path.join(streams_dir, 'PROCESSED', f'5haStreams_0.5m_buf{buffer_distance}.tif')
ditches_fp = os.path.join(ditches_dir, '5haStreams_ManuallyDigitizedDitchesKrycklan_merged_5.shp')
out_fp_ditches = os.path.join(ditches_dir, f'5haStreams_ManuallyDigitizedDitchesKrycklan_merged_5_0.5m_buf{buffer_distance}.tif')

# Buffer and rasterize streams
streams = gpd.read_file(streams_fp)
streams_buffered = streams.copy()
streams_buffered['geometry'] = streams.geometry.buffer(buffer_distance)
streams_buffered_fp = streams_fp.replace('.shp', f'_buf{buffer_distance}m.shp')
streams_buffered.to_file(streams_buffered_fp)

tools.rasterize_shapefile(
    shapefile=streams_buffered_fp,
    burn_field='GRID_CODE',
    subset=None,
    out_fp=out_fp_streams,
    ref_raster=cleaned_dem_fp,
    plot=False,
    save_in='geotiff'
)

# Buffer and rasterize ditches
ditches = gpd.read_file(ditches_fp)
ditches_buffered = ditches.copy()
ditches_buffered['geometry'] = ditches.geometry.buffer(buffer_distance)
ditches_buffered_fp = ditches_fp.replace('.shp', f'_buf{buffer_distance}m.shp')
ditches_buffered.to_file(ditches_buffered_fp)

tools.rasterize_shapefile(
    shapefile=ditches_buffered_fp,
    burn_field='VALUE',
    subset=None,
    out_fp=out_fp_ditches,
    ref_raster=cleaned_dem_fp,
    plot=False,
    save_in='geotiff'
)

for src_fp, file_end in [(out_fp_ditches, 'channels'), (out_fp_streams, '5haStreams')]:
    binary_fp = src_fp.replace('.tif', '_binary.tif')

    with rasterio.open(src_fp) as src:
        arr = src.read(1)
        profile = src.profile
        binary = np.where(arr > 0, 1, 0).astype(np.uint8)
        profile.update(dtype=rasterio.uint8, nodata=255)

    with rasterio.open(binary_fp, 'w', **profile) as dst:
        dst.write(binary, 1)

    out_dist = os.path.join(streams_dir, 'PROCESSED', f'{file_end}_distances.tif')
    wbt.euclidean_distance(i=binary_fp, output=out_dist)
    print(f'Wrote 0.5 m distance raster: {out_dist}')

/opt/miniconda3/envs/wbt/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field created_da create as date field, though DateTime requested.
  ogr_write(
/opt/miniconda3/envs/wbt/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field last_edite create as date field, though DateTime requested.
  ogr_write(


./whitebox_tools --run="EuclideanDistance" --input='/Users/jpnousu/Library/CloudStorage/OneDrive-Luonnonvarakeskus/Krycklan data/GIS/CATCHMENTS/ManualStreams/PROCESSED/5haStreams_ManuallyDigitizedDitchesKrycklan_merged_5_0.5m_buf0.5_binary.tif' --output='/Users/jpnousu/Library/CloudStorage/OneDrive-Luonnonvarakeskus/Krycklan data/GIS/CATCHMENTS/Krycklan_5ha_streams/PROCESSED/channels_distances.tif' -v --compress_rasters=False

********************************
* Welcome to EuclideanDistance *
* Powered by WhiteboxTools     *
* www.whiteboxgeo.com          *
********************************
Reading data...
Initializing Rasters: 0%
Initializing Rasters: 1%
Initializing Rasters: 2%
Initializing Rasters: 3%
Initializing Rasters: 4%
Initializing Rasters: 5%
Initializing Rasters: 6%
Initializing Rasters: 7%
Initializing Rasters: 8%
Initializing Rasters: 9%
Initializing Rasters: 10%
Initializing Rasters: 11%
Initializing Rasters: 12%
Initializing Rasters: 13%
Initializing Rasters: 14%
Initiali

In [3]:
# Step 1: Resample 0.5 m distances to 20 m and mask outside 20 m stream cells
for file_end in file_ends:
    dist_05m_fp = os.path.join(
        krycklan_data_path,
        'GIS', 'CATCHMENTS', 'Krycklan_5ha_streams', 'PROCESSED',
        f'{file_end}_distances.tif'
    )

    if not os.path.exists(dist_05m_fp):
        print(f'Missing 0.5 m distance raster: {dist_05m_fp}')
        continue

    for catch in catchs:
        outdir20 = os.path.join(krycklan_gis_data_path, '20m', f'C{catch}')
        os.makedirs(outdir20, exist_ok=True)

        ref20_fp = os.path.join(outdir20, 'dem_clip.asc')
        stream20_fp = os.path.join(outdir20, f'{file_end}.asc')
        dist20_raw_fp = os.path.join(outdir20, f'{file_end}_distance_raw20.asc')
        dist20_masked_fp = os.path.join(outdir20, f'{file_end}_distance.asc')
        valid20_fp = os.path.join(outdir20, f'{file_end}_valid20.asc')

        if not os.path.exists(ref20_fp) or not os.path.exists(stream20_fp):
            print(f'Skipping C{catch}, {file_end}: missing 20 m reference or stream mask')
            continue

        tools.reproj_match(
            infile=dist_05m_fp,
            match=ref20_fp,
            outfile=dist20_raw_fp,
            resampling_method='average',
            save_in='asc'
        )

        with rasterio.open(dist20_raw_fp) as dist_src, rasterio.open(stream20_fp) as stream_src:
            dist20 = dist_src.read(1).astype(np.float32)
            profile = dist_src.profile.copy()

            stream = stream_src.read(1)
            stream_nodata = stream_src.nodata

            valid_stream = np.isfinite(stream) & (stream > 0)
            if stream_nodata is not None:
                valid_stream &= (stream != stream_nodata)

            dist20[~valid_stream] = nodata_val
            profile.update(dtype=rasterio.float32, nodata=nodata_val, count=1)

        with rasterio.open(dist20_masked_fp, 'w', **profile) as dst:
            dst.write(dist20, 1)

        # Store support as float 0/1 to preserve fractional coverage when averaging to coarser grids.
        valid20 = np.where(dist20 != nodata_val, 1.0, 0.0).astype(np.float32)
        valid_profile = profile.copy()
        valid_profile.update(dtype=rasterio.float32, nodata=0.0, count=1)

        with rasterio.open(valid20_fp, 'w', **valid_profile) as dst:
            dst.write(valid20, 1)

        print(f'Wrote masked 20 m distance: {dist20_masked_fp}')

Coregistered to shape: 69 41 
 Affine | 20.00, 0.00, 732306.00|
| 0.00,-20.00, 7134870.00|
| 0.00, 0.00, 1.00|
Wrote masked 20 m distance: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C1/channels_distance.asc
Coregistered to shape: 39 30 
 Affine | 20.00, 0.00, 730786.00|
| 0.00,-20.00, 7134570.00|
| 0.00, 0.00, 1.00|
Wrote masked 20 m distance: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C2/channels_distance.asc
Coregistered to shape: 13 21 
 Affine | 20.00, 0.00, 731326.00|
| 0.00,-20.00, 7134490.00|
| 0.00, 0.00, 1.00|
Wrote masked 20 m distance: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C3/channels_distance.asc
Coregistered to shape: 32 25 
 Affine | 20.00, 0.00, 730766.00|
| 0.00,-20.00, 7135130.00|
| 0.00, 0.00, 1.00|
Wrote masked 20 m distance: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C4/channels_distance.asc
Coregistered to shape: 77 63 
 Affine | 20.00, 0.00, 729606.00|
| 0.00,-20.00, 7136110.00|
| 0.00, 0.00, 1.00|
Wrote masked 20 m distance: /Users/jpnousu/Data/Krycklan_GIS_data

In [4]:
# Step 2: Resample 0.5 m distances directly to each coarse grid and apply its stream mask
for file_end in file_ends:
    dist_05m_fp = os.path.join(
        krycklan_data_path,
        'GIS', 'CATCHMENTS', 'Krycklan_5ha_streams', 'PROCESSED',
        f'{file_end}_distances.tif'
    )

    if not os.path.exists(dist_05m_fp):
        print(f'Missing 0.5 m distance raster: {dist_05m_fp}')
        continue

    for catch in catchs:
        for reso in coarse_resos:
            outdir = os.path.join(krycklan_gis_data_path, f'{reso}m', f'C{catch}')
            os.makedirs(outdir, exist_ok=True)

            ref_fp = os.path.join(outdir, 'dem_clip.asc')
            stream_fp = os.path.join(outdir, f'{file_end}.asc')
            raw_dist_fp = os.path.join(outdir, f'{file_end}_distance_raw{reso}.asc')
            out_dist_fp = os.path.join(outdir, f'{file_end}_distance.asc')

            if not os.path.exists(ref_fp) or not os.path.exists(stream_fp):
                print(f'Skipping C{catch}, {file_end}, {reso}m: missing reference or stream mask')
                continue

            tools.reproj_match(
                infile=dist_05m_fp,
                match=ref_fp,
                outfile=raw_dist_fp,
                resampling_method='average',
                save_in='asc'
            )

            with rasterio.open(raw_dist_fp) as dist_src, rasterio.open(stream_fp) as stream_src:
                dist = dist_src.read(1).astype(np.float32)
                profile = dist_src.profile.copy()

                stream = stream_src.read(1)
                stream_nodata = stream_src.nodata

                valid_stream = np.isfinite(stream) & (stream > 0)
                if stream_nodata is not None:
                    valid_stream &= (stream != stream_nodata)

                dist[~valid_stream] = nodata_val
                profile.update(dtype=rasterio.float32, nodata=nodata_val, count=1)

            with rasterio.open(out_dist_fp, 'w', **profile) as dst:
                dst.write(dist, 1)

            print(f'Wrote {reso} m distance from 0.5 m source: {out_dist_fp}')

Coregistered to shape: 35 21 
 Affine | 40.00, 0.00, 732286.00|
| 0.00,-40.00, 7134870.00|
| 0.00, 0.00, 1.00|
Wrote 40 m distance from 0.5 m source: /Users/jpnousu/Data/Krycklan_GIS_data/40m/C1/channels_distance.asc
Coregistered to shape: 18 11 
 Affine | 80.00, 0.00, 732246.00|
| 0.00,-80.00, 7134870.00|
| 0.00, 0.00, 1.00|
Wrote 80 m distance from 0.5 m source: /Users/jpnousu/Data/Krycklan_GIS_data/80m/C1/channels_distance.asc
Coregistered to shape: 9 6 
 Affine | 160.00, 0.00, 732166.00|
| 0.00,-160.00, 7134870.00|
| 0.00, 0.00, 1.00|
Wrote 160 m distance from 0.5 m source: /Users/jpnousu/Data/Krycklan_GIS_data/160m/C1/channels_distance.asc
Coregistered to shape: 20 16 
 Affine | 40.00, 0.00, 730766.00|
| 0.00,-40.00, 7134590.00|
| 0.00, 0.00, 1.00|
Wrote 40 m distance from 0.5 m source: /Users/jpnousu/Data/Krycklan_GIS_data/40m/C2/channels_distance.asc
Coregistered to shape: 11 9 
 Affine | 80.00, 0.00, 730726.00|
| 0.00,-80.00, 7134630.00|
| 0.00, 0.00, 1.00|
Wrote 80 m distance 

In [5]:
# Step 3: Additional stream metrics (length, width, area)
from shapely.geometry import box

# Length settings
shapefiles = [
    os.path.join(krycklan_data_path, 'GIS', 'CATCHMENTS', 'Krycklan_5ha_streams', '5haStreams.shp'),
    os.path.join(krycklan_data_path, 'GIS', 'CATCHMENTS', 'ManualStreams', 'PROCESSED', '5haStreams_ManuallyDigitizedDitchesKrycklan_merged_5.shp'),
]
length_file_ends = ['5haStreams', 'channels']
all_resos = [20, 40, 80, 160]

# -----------------------------
# 3A) Length raster (line length inside each raster cell)
# -----------------------------
for shp_fp, file_end in zip(shapefiles, length_file_ends):
    gdf = gpd.read_file(shp_fp).to_crs(epsg=3006)

    for catch in catchs:
        for reso in all_resos:
            outdir = os.path.join(krycklan_gis_data_path, f'{reso}m', f'C{catch}')
            os.makedirs(outdir, exist_ok=True)

            ref_raster_fp = os.path.join(outdir, f'{file_end}.asc')

            if not os.path.exists(ref_raster_fp):
                print(f"Missing reference raster: {ref_raster_fp}")
                continue

            with rasterio.open(ref_raster_fp) as ref:
                profile = ref.profile.copy()
                profile.update(dtype='float32', nodata=nodata_val)
                transform = ref.transform
                width, height = ref.width, ref.height
                bounds = ref.bounds

            bbox = box(*bounds)
            gdf_clip = gdf.clip(bbox)
            arr = np.full((height, width), nodata_val, dtype='float32')

            if not gdf_clip.empty:
                for row in range(height):
                    for col in range(width):
                        x_min, y_max = transform * (col, row)
                        x_max = x_min + transform.a
                        y_min = y_max + transform.e
                        cell_poly = box(x_min, y_min, x_max, y_max)

                        inter = gdf_clip.intersection(cell_poly)
                        total_length = sum(geom.length for geom in inter if not geom.is_empty)

                        if total_length > 0:
                            arr[row, col] = total_length

            with rasterio.open(ref_raster_fp) as mask_src:
                mask_data = mask_src.read(1)
                mask_nodata = mask_src.nodata

            if mask_nodata is not None:
                arr[mask_data == mask_nodata] = nodata_val

            out_fp = os.path.join(outdir, f'{file_end}_length.asc')
            with rasterio.open(out_fp, 'w', **profile) as dst:
                dst.write(arr, 1)

            print(f"Wrote length raster: {out_fp}")

# -----------------------------
# 3B) Width raster from distance support
# -----------------------------
for reso in all_resos:
    for catch in catchs:
        outdir = os.path.join(krycklan_gis_data_path, f'{reso}m', f'C{catch}')

        ch_dist_file = os.path.join(outdir, 'channels_distance.asc')
        st_dist_file = os.path.join(outdir, '5haStreams_distance.asc')
        channels_out = os.path.join(outdir, 'channels_width.asc')
        streams_out = os.path.join(outdir, '5haStreams_width.asc')

        if not (os.path.exists(ch_dist_file) and os.path.exists(st_dist_file)):
            continue

        with rasterio.open(ch_dist_file) as ch_src, rasterio.open(st_dist_file) as st_src:
            ch_dist = ch_src.read(1)
            st_dist = st_src.read(1)

            valid_channels = (ch_dist != ch_src.nodata)
            valid_streams = (st_dist != st_src.nodata)

            streams_width = np.full(st_dist.shape, nodata_val, dtype=np.int16)
            streams_width[valid_streams] = 2

            st_profile = st_src.profile.copy()
            st_profile.update(dtype=rasterio.int16, nodata=nodata_val)
            with rasterio.open(streams_out, 'w', **st_profile) as dst:
                dst.write(streams_width, 1)

            channels_width = np.full(ch_dist.shape, nodata_val, dtype=np.int16)
            channels_width[valid_channels] = 1
            overlap = valid_channels & valid_streams
            channels_width[overlap] = 2

            ch_profile = ch_src.profile.copy()
            ch_profile.update(dtype=rasterio.int16, nodata=nodata_val)
            with rasterio.open(channels_out, 'w', **ch_profile) as dst:
                dst.write(channels_width, 1)

        print(f"Wrote width rasters: {channels_out}, {streams_out}")

# -----------------------------
# 3C) Area raster = length * width (optional normalization)
# -----------------------------
normalize = True

for reso in all_resos:
    cell_area = float(reso * reso)

    for catch in catchs:
        outdir = os.path.join(krycklan_gis_data_path, f'{reso}m', f'C{catch}')

        for prefix in ['channels', '5haStreams']:
            length_file = os.path.join(outdir, f'{prefix}_length.asc')
            width_file = os.path.join(outdir, f'{prefix}_width.asc')
            out_file = os.path.join(outdir, f'{prefix}_area.asc')

            if not (os.path.exists(length_file) and os.path.exists(width_file)):
                continue

            with rasterio.open(length_file) as len_src, rasterio.open(width_file) as wid_src:
                length = len_src.read(1).astype(float)
                width = wid_src.read(1).astype(float)

                valid = (length != len_src.nodata) & (width != wid_src.nodata)
                area = np.full(length.shape, nodata_val, dtype=np.float32)

                area[valid] = length[valid] * width[valid]
                if normalize:
                    area[valid] = area[valid] / cell_area

                profile = len_src.profile.copy()
                profile.update(dtype=rasterio.float32, nodata=nodata_val)

            with rasterio.open(out_file, 'w', **profile) as dst:
                dst.write(area, 1)

            print(f"Wrote area raster: {out_file}")

Wrote length raster: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C1/5haStreams_length.asc
Wrote length raster: /Users/jpnousu/Data/Krycklan_GIS_data/40m/C1/5haStreams_length.asc
Wrote length raster: /Users/jpnousu/Data/Krycklan_GIS_data/80m/C1/5haStreams_length.asc
Wrote length raster: /Users/jpnousu/Data/Krycklan_GIS_data/160m/C1/5haStreams_length.asc
Wrote length raster: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C2/5haStreams_length.asc
Wrote length raster: /Users/jpnousu/Data/Krycklan_GIS_data/40m/C2/5haStreams_length.asc
Wrote length raster: /Users/jpnousu/Data/Krycklan_GIS_data/80m/C2/5haStreams_length.asc
Wrote length raster: /Users/jpnousu/Data/Krycklan_GIS_data/160m/C2/5haStreams_length.asc
Wrote length raster: /Users/jpnousu/Data/Krycklan_GIS_data/20m/C3/5haStreams_length.asc
Wrote length raster: /Users/jpnousu/Data/Krycklan_GIS_data/40m/C3/5haStreams_length.asc
Wrote length raster: /Users/jpnousu/Data/Krycklan_GIS_data/80m/C3/5haStreams_length.asc
Wrote length raster: /Users/jp